## 1.数据读取与预处理

In [ ]:
with open('data/FinancialPhraseBank/Sentences_75Agree.txt', 'r') as f: 
    FPB_75_in = f.read() #Malo数据特色，分为不同置信度的样本
with open('data/FinancialPhraseBank/Sentences_AllAgree.txt', 'r') as f:
    FPB_100_in = f.read()

In [ ]:
FPB_75_in[:200]

In [ ]:
FPB_75_lines = FPB_75_in.split('\n')

In [ ]:
FPB_75_lines[0].split('@')

In [ ]:
from string import punctuation

FPB_75_data = []
FPB_75_label = []
tags = {'negative':0,'neutral':1,'positive':2}
for line in FPB_75_lines[:-1]:
    line = line.lower()
    data_line = line.split('@')[0]
    FPB_75_label.append(tags[line.split('@')[1]])
    data_line = ''.join([c for c in data_line if c not in punctuation]) #去除标点
    data_l = data_line.split(' ')
    data_l = [c for c in data_l if len(c)>0] #Malo这份数据特别之处，标点前有空格，故而去除标点后再以空格区分，出现了一些空值
    FPB_75_data.append(data_l)
    

In [ ]:
print(FPB_75_data[0],'\n',FPB_75_label[0])

In [ ]:
FPB_100_lines = FPB_100_in.split('\n')
FPB_100_data = []
FPB_100_label = []
tags = {'negative':0,'neutral':1,'positive':2}
for line in FPB_100_lines[:-1]:
    line = line.lower()
    data_line = line.split('@')[0]
    FPB_100_label.append(tags[line.split('@')[1]])
    data_line = ''.join([c for c in data_line if c not in punctuation])
    data_l = data_line.split(' ')
    data_l = [c for c in data_l if len(c)>0]
    FPB_100_data.append(data_l)

In [ ]:
FPB_data_raw = FPB_75_data + FPB_100_data
FPB_label_raw = FPB_75_label + FPB_100_label

In [ ]:
FP_len = []
for li in FPB_data_raw:
    FP_len.append(len(li))

import pandas as pd

pd.DataFrame(FP_len).describe()

In [ ]:
import random 
random.seed(42)
raw_idx = list(range(len(FPB_data_raw)))

In [ ]:
random.shuffle(raw_idx)


In [ ]:
FPB_data = []
FPB_label = []
for idx in raw_idx:
    FPB_data.append(FPB_data_raw[idx])
    FPB_label.append(FPB_label_raw[idx])

In [ ]:
FPB_label_raw[0]

## 2.加载预训练的word2vec模型

In [ ]:
# 使用gensim模块来加载word2vec model
from gensim.models import KeyedVectors
# 加载模型，一定加载SLIM模型，加载完整版本，内存/显存占据过大
# 下载链接：https://github.com/eyaler/word2vec-slim/blob/master/GoogleNews-vectors-negative300-SLIM.bin.gz
embed_lookup = KeyedVectors.load_word2vec_format('model/GoogleNews-vectors-negative300-SLIM.bin', 
                                                 binary=True)

In [ ]:
embed_lookup.key_to_index['hello']

In [ ]:
def get_vec_index(wordl): #本函数获得所有词的word2vec的index
    resl = []
    for wi in wordl:
        try:
            idx = embed_lookup.key_to_index[wi]
        except Exception:
            idx = 0
        resl.append(idx)
    return resl

In [ ]:
FPB_vec_index = list(map(get_vec_index,FPB_data))

In [ ]:
import numpy as np
def padding_tokens(raw_vec,padding_len=60, sq_len=None): #对所有数据进行补齐，在本例中数据长度都小于60，另外一些情况下，需要对数据进行截断
    tlen = len(raw_vec)
    if sq_len is not None:
        sq_len.append(tlen)
    if tlen>=padding_len:
        return raw_vec[:padding_len]
    
    for li in range(padding_len-tlen):
        raw_vec.append(0)
    return np.array(raw_vec)

In [ ]:
sq_len = []
FPB_vec_index_padding = list(map(lambda x: padding_tokens(x, padding_len=60, sq_len=sq_len),FPB_vec_index))


In [ ]:
features = np.array(FPB_vec_index_padding)

In [ ]:
labels = np.array(FPB_label)

In [ ]:
features[:10,:6]

## 3.标准化数据

In [ ]:
#切分训练集、验证集、测试集
train_frac = 0.8
valid_frac = 0.9

train_idx = int(len(features)*train_frac)
valid_idx = int(len(features)*valid_frac)

train_x, valid_x,test_x = features[:train_idx], features[train_idx:valid_idx], features[valid_idx:]
train_y, valid_y,test_y = labels[:train_idx], labels[train_idx:valid_idx], labels[valid_idx:]
train_sq_len, valid_sq_len, test_sq_len = sq_len[:train_idx], sq_len[train_idx:valid_idx], sq_len[valid_idx:]


In [ ]:
import torch
from torch.utils.data import TensorDataset, DataLoader

# 制作标准Tensor数据集，包含特征、标签
train_data = TensorDataset(torch.from_numpy(train_x), torch.from_numpy(train_y))
valid_data = TensorDataset(torch.from_numpy(valid_x), torch.from_numpy(valid_y))
test_data = TensorDataset(torch.from_numpy(test_x), torch.from_numpy(test_y))

batch_size = 50

# 随机化与Batch
train_loader = DataLoader(train_data, shuffle=True, batch_size=batch_size)
valid_loader = DataLoader(valid_data, shuffle=True, batch_size=batch_size)
test_loader = DataLoader(test_data, shuffle=True, batch_size=batch_size)

In [ ]:
#保存数据，以供后期使用
torch.save({"data": train_data.tensors[0], "labels": train_data.tensors[1], "lengths": torch.tensor(train_sq_len)}, "data/Malo/Malo_Train.pth")
torch.save({"data": valid_data.tensors[0], "labels": valid_data.tensors[1], "lengths": torch.tensor(valid_sq_len)}, "data/Malo/Malo_Valid.pth")
torch.save({"data": test_data.tensors[0], "labels": test_data.tensors[1], "lengths": torch.tensor(test_sq_len)}, "data/Malo/Malo_Test.pth")

In [ ]:
len(valid_x)

In [ ]:
# 检查 DataLoader 的输出
for batch_idx, (x, y) in enumerate(train_loader):
    print(f"训练集 - Batch {batch_idx}")
    print(f"Features shape: {x.shape}")  # [batch_size, word_len]
    print(f"Labels shape: {y.shape}")    # [batch_size]
    break  # 查看第一个批次的结果即可

## 4.定义模型

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class TextCNN(nn.Module):
    
    def __init__(self, embed_model, vocab_size, output_size, embed_dim,    #在init函数中注明初始化class TextCNN时需要给定的参数
                 num_filters=100, kernel_sizes=[3, 4, 5], embed_freeze=True, drop_prob=0.5):
        
        super(TextCNN, self).__init__()

        #设定参数
        self.num_filters = num_filters
        self.embed_dim = embed_dim
        
        # 1.嵌入层
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        # 载入预训练的权重
        self.embedding.weight = nn.Parameter(torch.from_numpy(embed_model.vectors)) # all vectors
        # 冻结嵌入层的训练，设定为false则一起训练
        if embed_freeze:
            self.embedding.requires_grad = False
        
        # 2.卷积层
        self.convs_1d = nn.ModuleList([
            nn.Conv2d(1, num_filters, (k, embed_dim), padding=(k-2,0))  #并联多个二维卷积层，每个卷积核的宽度设定为embed_dim
            for k in kernel_sizes])
        
        # 3. 全连接层
        self.fc = nn.Linear(len(kernel_sizes) * num_filters, output_size) 
        
        # 4. drop_out
        self.dropout = nn.Dropout(drop_prob)
        
        
    
    def conv_and_pool(self, x, conv):  #定义一个函数，用于卷积计算后的pooling和数据变形
       
        x = F.relu(conv(x)).squeeze(3)
        #这是一个函数，对于每一个conv（一维卷积），卷积后的结果第四个维度会是1，可以删掉（squeeze）（四个维度：（batch_size,num_filters,sequence_length,1)）
        x_max = F.max_pool1d(x, x.size(2)).squeeze(2)
        #一维池化，第二个参数是第三个维度，即sequence_length，池化完毕后，会变成（batch_size,num_filters,1)，然后再删掉第三个维度
        return x_max
        #返回的形状是（batch_size,num_filters)

    def forward(self, x):
        
        # 嵌入层转化为vectors
        embeds = self.embedding(x) 
        # 输入x: (batch_size, seq_length)，其中 seq_length 是输入序列的长度。
        # 输出embeds: (batch_size, seq_length, embedding_dim)，嵌入向量的每个元素维度为 embedding_dim
        
        # unsqueeze(1) 在第 1 维插入一个大小为 1 的通道维度，目的是适配卷积层的输入要求。
        embeds = embeds.unsqueeze(1)
        # 输入形状 embeds: (batch_size, seq_length, embedding_dim)
        # 输出形状 embeds: (batch_size, 1, seq_length, embedding_dim) 1是单通道
        
        # 一个list，存储每个conv的结果
        conv_results = [self.conv_and_pool(embeds, conv) for conv in self.convs_1d]
        #self.convs_1d 是一个包含多个 2D 卷积核的 ModuleList，每个卷积核负责不同大小的窗口（kernel_size）。
        # 对每个卷积核 conv：
        #    进行卷积操作，通过 self.conv_and_pool 提取特征。
        #    每个卷积核会生成一个形状为 (batch_size, num_filters) 的特征图。
    	# 最终 conv_results 是一个列表，包含所有卷积核的输出。

        # 拼接conv的结果
        x = torch.cat(conv_results, 1)
        #添加dropout
        x = self.dropout(x)

        #全连接
        x = self.fc(x)

        #softmax输出
        return F.log_softmax(x, dim=1)
      

## 5.定义训练、测试函数

In [ ]:
#训练函数与测试函数仍然可以复用（why？因为依然是分类问题）
#定义一个训练函数
def train(args, model, device, train_loader, optimizer, epoch):
    # args：一般是一个包含超参数的对象或命名空间（如 batch_size, learning_rate 等），用于控制训练过程。
    # model：传入的神经网络模型，需要被训练。
    # device：设备信息（如 cpu 或 cuda），决定模型和数据在哪个硬件上运行。
    # train_loader：数据加载器，包含训练数据的批次，用于迭代访问每个 batch。
    # optimizer：优化器，用于更新模型的参数（如 SGD、Adam）。
    # epoch：当前的训练轮次编号，用于记录进度。
    
    model.train()  #将模型切换为训练模式。pytorch中有train和eval两个模式，一些机制如dropout Batchnorm仅在train模式中生效
    for batch_idx, (data, target) in enumerate(train_loader):  #从train_loader中加载数据，data、target分别为数据和标签，batch_idx为批次索引从0开始
        data, target = data.to(device), target.to(device)  #将数据和标签转移到指定的设备（CPU 或 GPU）。如果数据和标签不在相同设备上则会报错
        optimizer.zero_grad()     #清空优化器中所有参数的梯度缓存。PyTorch 的默认行为是 梯度累积（每次调用 backward() 会将新梯度累加到现有的梯度上），因此需要在每次迭代开始前清空梯度。
        output = model(data)      #使用模型对当前批次的输入 data 进行前向传播，得到预测结果 output
        loss = F.nll_loss(output, target)  #计算预测值与真实标签之间的误差（损失）：F.nll_loss是 PyTorch 提供的负对数似然损失函数（Negative Log Likelihood Loss）
        loss.backward()           #loss反向传播
        optimizer.step()          #根据计算得到的梯度，利用优化器更新模型的参数

#定义一个测试函数
def test(model, device, test_loader):
    # model：需要评估的神经网络模型。
    # device：用于运行模型和数据的设备（如 CPU 或 GPU）。
    # test_loader：数据加载器，提供测试数据集，通常不会打乱顺序。
    
    model.eval()  #将模型切换为eval模式
    test_loss = 0
    correct = 0
    with torch.no_grad():  #禁用自动求导功能，节省资源，加速运行
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)    #将数据和标签转移到指定的设备（CPU 或 GPU）。如果数据和标签不在相同设备上则会报错
            output = model(data)    #将数据输入模型，得到预测结果 output
            test_loss += F.nll_loss(output, target, reduction='sum').item()  #使用负对数似然损失（nll_loss）计算当前批次的损失，并将其加总
            pred = output.argmax(dim=1, keepdim=True)   #从模型输出中选择每个样本中最大值对应的索引作为预测类别。
                                                        #argmax(dim=1)：在每行（样本）的预测值中找到概率最大的类别索引。
                                                        #keepdim=True：保持结果的维度一致，方便与标签进行比较。
            correct += pred.eq(target.view_as(pred)).sum().item()  #累计正确预测数量；pred.eq(target.view_as(pred))：比较预测值 pred 和真实值 target 是否相等，返回一个布尔张量。

    test_loss /= len(test_loader.dataset)  #总测试损失除以测试集样本总数，得到平均损失。

    print("=" * 50)
    print("Evaluation Results:")
    print(f" - Average Loss: {test_loss:.4f}")
    print(f" - Accuracy: {correct}/{len(test_loader.dataset)} "
          f"({100. * correct / len(test_loader.dataset):.2f}%)")
    print("=" * 50)


In [ ]:
epochs = 6  #定义最大训练周期
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")  #定义设备,CPU或者GPU（Cuda）
print(device)

In [ ]:
pretrained_words = []
for word in embed_lookup.key_to_index:
    pretrained_words.append(word)

In [ ]:

vocab_size = len(pretrained_words)
output_size = 3 
embedding_dim = len(embed_lookup[pretrained_words[0]]) # 300-dim vectors
num_filters = 100
kernel_sizes = [3, 4, 5]

model = TextCNN(embed_lookup, vocab_size, output_size, embedding_dim,
                   num_filters, kernel_sizes).to(device)
print(model)


In [ ]:
import torch.optim as optim
optimizer = optim.Adadelta(model.parameters()) #Adadelta 是一种自适应学习率优化算法，不需要显式设置全局学习率

## 6.Action

In [ ]:
import time
for epoch in range(1, epochs + 1):
    start_time = time.time()  #记录开始时间
    train([], model, device, train_loader, optimizer, epoch)  #训练
    train_time = time.time()
    test(model, device, valid_loader)  #测试
    test_time = time.time()
    train_duration = train_time - start_time
    test_duration = test_time - train_time
    print(f"Epoch {epoch}/{epochs} - Trained in {train_duration:.2f} seconds  - Test in {test_duration:.2f} seconds.\n")


In [ ]:
test(model, device, valid_loader)

In [ ]:
torch.save(model.state_dict(), "model/TextCNN.pth")